In [3]:
# ============================================================
# CELL 0: IMPORTS AND CONFIG
# ============================================================
# This cell sets up the tools and defines which dataset you're using.
# For a new project, you only need to change the CONFIG section.
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Set display options so we can see more rows/columns
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 150)
pd.set_option('display.width', 200)

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

/kaggle/input/competitions/house-prices-advanced-regression-techniques/sample_submission.csv
/kaggle/input/competitions/house-prices-advanced-regression-techniques/data_description.txt
/kaggle/input/competitions/house-prices-advanced-regression-techniques/train.csv
/kaggle/input/competitions/house-prices-advanced-regression-techniques/test.csv


In [6]:
# --- CONFIG: Change these for each new project ---
TRAIN_PATH = '/kaggle/input/competitions/house-prices-advanced-regression-techniques/train.csv'
TEST_PATH = '/kaggle/input/competitions/house-prices-advanced-regression-techniques/test.csv'
DESCRIPTION_PATH = '/kaggle/input/competitions/house-prices-advanced-regression-techniques/data_description.txt'  # Optional: some competitions provide this

TARGET_COL = 'SalePrice'  # The column you are trying to predict
ID_COL = 'Id'             # The row identifier (not a feature)

# If you don't have a description file, set this to None
# and skip the description-reading cells.
# ============================================================

print("Imports complete. Ready to load data.")

Imports complete. Ready to load data.


In [7]:
# ============================================================
# CELL 1: LOAD THE DATA
# ============================================================
# WHAT THIS DOES:
#   Reads train.csv and test.csv into pandas DataFrames.
#
# WHY THIS MATTERS:
#   In supervised learning, you always have at least two files:
#   one with the target (train) and one without (test).
#   Keeping them separate at first prevents accidentally leaking
#   target information into your exploration.
#
# WHAT TO ASK YOURSELF:
#   - Did both files load successfully?
#   - How many rows and columns does each have?
#   - Does the test set have one fewer column? (It should — the target.)

train_df = pd.read_csv(TRAIN_PATH)
test_df = pd.read_csv(TEST_PATH)

print("=== SHAPES ===")
print(f"Train shape: {train_df.shape}")
print(f"Test shape:  {test_df.shape}")

print("\n=== COLUMN NAMES ===")
print(f"Train has {len(train_df.columns)} columns:")
print(list(train_df.columns))

print("\n=== FIRST 5 ROWS OF TRAIN ===")
display(train_df.head())

print("\n=== INFO ===")
train_df.info()


=== SHAPES ===
Train shape: (1460, 81)
Test shape:  (1459, 80)

=== COLUMN NAMES ===
Train has 81 columns:
['Id', 'MSSubClass', 'MSZoning', 'LotFrontage', 'LotArea', 'Street', 'Alley', 'LotShape', 'LandContour', 'Utilities', 'LotConfig', 'LandSlope', 'Neighborhood', 'Condition1', 'Condition2', 'BldgType', 'HouseStyle', 'OverallQual', 'OverallCond', 'YearBuilt', 'YearRemodAdd', 'RoofStyle', 'RoofMatl', 'Exterior1st', 'Exterior2nd', 'MasVnrType', 'MasVnrArea', 'ExterQual', 'ExterCond', 'Foundation', 'BsmtQual', 'BsmtCond', 'BsmtExposure', 'BsmtFinType1', 'BsmtFinSF1', 'BsmtFinType2', 'BsmtFinSF2', 'BsmtUnfSF', 'TotalBsmtSF', 'Heating', 'HeatingQC', 'CentralAir', 'Electrical', '1stFlrSF', '2ndFlrSF', 'LowQualFinSF', 'GrLivArea', 'BsmtFullBath', 'BsmtHalfBath', 'FullBath', 'HalfBath', 'BedroomAbvGr', 'KitchenAbvGr', 'KitchenQual', 'TotRmsAbvGrd', 'Functional', 'Fireplaces', 'FireplaceQu', 'GarageType', 'GarageYrBlt', 'GarageFinish', 'GarageCars', 'GarageArea', 'GarageQual', 'GarageCond', '

,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,LotConfig,LandSlope,Neighborhood,Condition1,Condition2,BldgType,HouseStyle,OverallQual,OverallCond,YearBuilt,YearRemodAdd,RoofStyle,RoofMatl,Exterior1st,Exterior2nd,MasVnrType,MasVnrArea,ExterQual,ExterCond,Foundation,BsmtQual,BsmtCond,BsmtExposure,BsmtFinType1,BsmtFinSF1,BsmtFinType2,BsmtFinSF2,BsmtUnfSF,TotalBsmtSF,Heating,HeatingQC,CentralAir,Electrical,1stFlrSF,2ndFlrSF,LowQualFinSF,GrLivArea,BsmtFullBath,BsmtHalfBath,FullBath,HalfBath,BedroomAbvGr,KitchenAbvGr,KitchenQual,TotRmsAbvGrd,Functional,Fireplaces,FireplaceQu,GarageType,GarageYrBlt,GarageFinish,GarageCars,GarageArea,GarageQual,GarageCond,PavedDrive,WoodDeckSF,OpenPorchSF,EnclosedPorch,3SsnPorch,ScreenPorch,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice
0,1,60,RL,65.0,8450,Pave,NaN,Reg,Lvl,AllPub,Inside,Gtl,CollgCr,Norm,Norm,1Fam,2Story,7,5,2003,2003,Gable,CompShg,VinylSd,VinylSd,BrkFace,196.0,Gd,TA,PConc,Gd,TA,No,GLQ,706,Unf,0,150,856,GasA,Ex,Y,SBrkr,856,854,0,1710,1,0,2,1,3,1,Gd,8,Typ,0,NaN,Attchd,2003.0,RFn,2,548,TA,TA,Y,0,61,0,0,0,0,NaN,NaN,NaN,0,2,2008,WD,Normal,208500
1,2,20,RL,80.0,9600,Pave,NaN,Reg,Lvl,AllPub,FR2,Gtl,Veenker,Feedr,Norm,1Fam,1Story,6,8,1976,1976,Gable,CompShg,MetalSd,MetalSd,NaN,0.0,TA,TA,CBlock,Gd,TA,Gd,ALQ,978,Unf,0,284,1262,GasA,Ex,Y,SBrkr,1262,0,0,1262,0,1,2,0,3,1,TA,6,Typ,1,TA,Attchd,1976.0,RFn,2,460,TA,TA,Y,298,0,0,0,0,0,NaN,NaN,NaN,0,5,2007,WD,Normal,181500
2,3,60,RL,68.0,11250,Pave,NaN,IR1,Lvl,AllPub,Inside,Gtl,CollgCr,Norm,Norm,1Fam,2Story,7,5,2001,2002,Gable,CompShg,VinylSd,VinylSd,BrkFace,162.0,Gd,TA,PConc,Gd,TA,Mn,GLQ,486,Unf,0,434,920,GasA,Ex,Y,SBrkr,920,866,0,1786,1,0,2,1,3,1,Gd,6,Typ,1,TA,Attchd,2001.0,RFn,2,608,TA,TA,Y,0,42,0,0,0,0,NaN,NaN,NaN,0,9,2008,WD,Normal,223500
3,4,70,RL,60.0,9550,Pave,NaN,IR1,Lvl,AllPub,Corner,Gtl,Crawfor,Norm,Norm,1Fam,2Story,7,5,1915,1970,Gable,CompShg,Wd Sdng,Wd Shng,NaN,0.0,TA,TA,BrkTil,TA,Gd,No,ALQ,216,Unf,0,540,756,GasA,Gd,Y,SBrkr,961,756,0,1717,1,0,1,0,3,1,Gd,7,Typ,1,Gd,Detchd,1998.0,Unf,3,642,TA,TA,Y,0,35,272,0,0,0,NaN,NaN,NaN,0,2,2006,WD,Abnorml,140000
4,5,60,RL,84.0,14260,Pave,NaN,IR1,Lvl,AllPub,FR2,Gtl,NoRidge,Norm,Norm,1Fam,2Story,8,5,2000,2000,Gable,CompShg,VinylSd,VinylSd,BrkFace,350.0,Gd,TA,PConc,Gd,TA,Av,GLQ,655,Unf,0,490,1145,GasA,Ex,Y,SBrkr,1145,1053,0,2198,1,0,2,1,4,1,Gd,9,Typ,1,TA,Attchd,2000.0,RFn,3,836,TA,TA,Y,192,84,0,0,0,0,NaN,NaN,NaN,0,12,2008,WD,Normal,250000



=== INFO ===
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1460 entries, 0 to 1459
Data columns (total 81 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Id             1460 non-null   int64  
 1   MSSubClass     1460 non-null   int64  
 2   MSZoning       1460 non-null   object 
 3   LotFrontage    1201 non-null   float64
 4   LotArea        1460 non-null   int64  
 5   Street         1460 non-null   object 
 6   Alley          91 non-null     object 
 7   LotShape       1460 non-null   object 
 8   LandContour    1460 non-null   object 
 9   Utilities      1460 non-null   object 
 10  LotConfig      1460 non-null   object 
 11  LandSlope      1460 non-null   object 
 12  Neighborhood   1460 non-null   object 
 13  Condition1     1460 non-null   object 
 14  Condition2     1460 non-null   object 
 15  BldgType       1460 non-null   object 
 16  HouseStyle     1460 non-null   object 
 17  OverallQual    1460 non-null   int64  

In [8]:
# ============================================================
# CELL 2: CONFIRM TARGET AND ID COLUMNS
# ============================================================
# WHAT THIS DOES:
#   Verifies that the target column exists in train and is missing
#   from test. Also confirms the ID column.
#
# WHY THIS MATTERS:
#   You must know exactly which column is your prediction target
#   and which column is just a row identifier. If you accidentally
#   use the target as a feature, your model will look perfect but
#   fail completely on test data.
#
# WHAT TO CHECK:
#   - TARGET_COL should be in train but NOT in test.
#   - ID_COL should be in both train and test.
#   - There should be exactly one target column.

assert TARGET_COL in train_df.columns, f"Target column '{TARGET_COL}' not found in train!"
assert TARGET_COL not in test_df.columns, f"Target column '{TARGET_COL}' should not be in test!"
assert ID_COL in train_df.columns and ID_COL in test_df.columns, f"ID column '{ID_COL}' missing!"

print(f"✅ Target column:  '{TARGET_COL}'")
print(f"✅ ID column:      '{ID_COL}'")

# Store the target separately for later analysis
y_train = train_df[TARGET_COL].copy()

print(f"\nTarget statistics:")
print(y_train.describe())


✅ Target column:  'SalePrice'
✅ ID column:      'Id'

Target statistics:
count      1460.000000
mean     180921.195890
std       79442.502883
min       34900.000000
25%      129975.000000
50%      163000.000000
75%      214000.000000
max      755000.000000
Name: SalePrice, dtype: float64


In [26]:
train_df.head()

,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,LotConfig,LandSlope,Neighborhood,Condition1,Condition2,BldgType,HouseStyle,OverallQual,OverallCond,YearBuilt,YearRemodAdd,RoofStyle,RoofMatl,Exterior1st,Exterior2nd,MasVnrType,MasVnrArea,ExterQual,ExterCond,Foundation,BsmtQual,BsmtCond,BsmtExposure,BsmtFinType1,BsmtFinSF1,BsmtFinType2,BsmtFinSF2,BsmtUnfSF,TotalBsmtSF,Heating,HeatingQC,CentralAir,Electrical,1stFlrSF,2ndFlrSF,LowQualFinSF,GrLivArea,BsmtFullBath,BsmtHalfBath,FullBath,HalfBath,BedroomAbvGr,KitchenAbvGr,KitchenQual,TotRmsAbvGrd,Functional,Fireplaces,FireplaceQu,GarageType,GarageYrBlt,GarageFinish,GarageCars,GarageArea,GarageQual,GarageCond,PavedDrive,WoodDeckSF,OpenPorchSF,EnclosedPorch,3SsnPorch,ScreenPorch,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice
0,1,60,RL,65.0,8450,Pave,NaN,Reg,Lvl,AllPub,Inside,Gtl,CollgCr,Norm,Norm,1Fam,2Story,7,5,2003,2003,Gable,CompShg,VinylSd,VinylSd,BrkFace,196.0,Gd,TA,PConc,Gd,TA,No,GLQ,706,Unf,0,150,856,GasA,Ex,Y,SBrkr,856,854,0,1710,1,0,2,1,3,1,Gd,8,Typ,0,NaN,Attchd,2003.0,RFn,2,548,TA,TA,Y,0,61,0,0,0,0,NaN,NaN,NaN,0,2,2008,WD,Normal,208500
1,2,20,RL,80.0,9600,Pave,NaN,Reg,Lvl,AllPub,FR2,Gtl,Veenker,Feedr,Norm,1Fam,1Story,6,8,1976,1976,Gable,CompShg,MetalSd,MetalSd,NaN,0.0,TA,TA,CBlock,Gd,TA,Gd,ALQ,978,Unf,0,284,1262,GasA,Ex,Y,SBrkr,1262,0,0,1262,0,1,2,0,3,1,TA,6,Typ,1,TA,Attchd,1976.0,RFn,2,460,TA,TA,Y,298,0,0,0,0,0,NaN,NaN,NaN,0,5,2007,WD,Normal,181500
2,3,60,RL,68.0,11250,Pave,NaN,IR1,Lvl,AllPub,Inside,Gtl,CollgCr,Norm,Norm,1Fam,2Story,7,5,2001,2002,Gable,CompShg,VinylSd,VinylSd,BrkFace,162.0,Gd,TA,PConc,Gd,TA,Mn,GLQ,486,Unf,0,434,920,GasA,Ex,Y,SBrkr,920,866,0,1786,1,0,2,1,3,1,Gd,6,Typ,1,TA,Attchd,2001.0,RFn,2,608,TA,TA,Y,0,42,0,0,0,0,NaN,NaN,NaN,0,9,2008,WD,Normal,223500
3,4,70,RL,60.0,9550,Pave,NaN,IR1,Lvl,AllPub,Corner,Gtl,Crawfor,Norm,Norm,1Fam,2Story,7,5,1915,1970,Gable,CompShg,Wd Sdng,Wd Shng,NaN,0.0,TA,TA,BrkTil,TA,Gd,No,ALQ,216,Unf,0,540,756,GasA,Gd,Y,SBrkr,961,756,0,1717,1,0,1,0,3,1,Gd,7,Typ,1,Gd,Detchd,1998.0,Unf,3,642,TA,TA,Y,0,35,272,0,0,0,NaN,NaN,NaN,0,2,2006,WD,Abnorml,140000
4,5,60,RL,84.0,14260,Pave,NaN,IR1,Lvl,AllPub,FR2,Gtl,NoRidge,Norm,Norm,1Fam,2Story,8,5,2000,2000,Gable,CompShg,VinylSd,VinylSd,BrkFace,350.0,Gd,TA,PConc,Gd,TA,Av,GLQ,655,Unf,0,490,1145,GasA,Ex,Y,SBrkr,1145,1053,0,2198,1,0,2,1,4,1,Gd,9,Typ,1,TA,Attchd,2000.0,RFn,3,836,TA,TA,Y,192,84,0,0,0,0,NaN,NaN,NaN,0,12,2008,WD,Normal,250000


In [29]:
# ============================================================
# CELL 3: FIRST-PASS SEPARATION BY DTYPE
# ============================================================
# WHAT THIS DOES:
#   Splits columns into numeric and non-numeric based on pandas dtypes.
#
# WHY THIS MATTERS:
#   This is your starting point, but it is NOT final. Pandas dtypes
#   can be misleading. A column stored as int might be a category code.
#   A column stored as object might contain numbers that were read as
#   strings.
#
# WHAT TO ASK YOURSELF:
#   - Are there any columns that look suspicious? (e.g., IDs stored as int)
#   - Are there object columns that should be numeric?
#   - Are there numeric columns that are actually categorical codes?

numeric_cols = train_df.select_dtypes(include=[np.number]).columns.tolist()
categorical_cols = train_df.select_dtypes(include=['object']).columns.tolist()

# Remove target and ID from the numeric list — they are not features
if TARGET_COL in numeric_cols:
    numeric_cols.remove(TARGET_COL)
if ID_COL in numeric_cols:
    numeric_cols.remove(ID_COL)

print(f"=== FIRST-PASS DTYPES ===")
print(f"Numeric columns ({len(numeric_cols)}):")
for i, col in enumerate(numeric_cols, 1):
    print(f"  {i:2d}. {col}")

print(f"\nCategorical/object columns ({len(categorical_cols)}:")
for i, col in enumerate(categorical_cols, 1):
    print(f"  {i:2d}. {col}")

print(f"\nTotal features: {len(numeric_cols) + len(categorical_cols)}")


=== FIRST-PASS DTYPES ===
Numeric columns (36):
   1. MSSubClass
   2. LotFrontage
   3. LotArea
   4. OverallQual
   5. OverallCond
   6. YearBuilt
   7. YearRemodAdd
   8. MasVnrArea
   9. BsmtFinSF1
  10. BsmtFinSF2
  11. BsmtUnfSF
  12. TotalBsmtSF
  13. 1stFlrSF
  14. 2ndFlrSF
  15. LowQualFinSF
  16. GrLivArea
  17. BsmtFullBath
  18. BsmtHalfBath
  19. FullBath
  20. HalfBath
  21. BedroomAbvGr
  22. KitchenAbvGr
  23. TotRmsAbvGrd
  24. Fireplaces
  25. GarageYrBlt
  26. GarageCars
  27. GarageArea
  28. WoodDeckSF
  29. OpenPorchSF
  30. EnclosedPorch
  31. 3SsnPorch
  32. ScreenPorch
  33. PoolArea
  34. MiscVal
  35. MoSold
  36. YrSold

Categorical/object columns (43:
   1. MSZoning
   2. Street
   3. Alley
   4. LotShape
   5. LandContour
   6. Utilities
   7. LotConfig
   8. LandSlope
   9. Neighborhood
  10. Condition1
  11. Condition2
  12. BldgType
  13. HouseStyle
  14. RoofStyle
  15. RoofMatl
  16. Exterior1st
  17. Exterior2nd
  18. MasVnrType
  19. ExterQual
  20.

In [35]:
# ============================================================
# CELL 4: INSPECT NUMERIC COLUMNS
# ============================================================
# WHAT THIS DOES:
#   For each numeric column, shows: number of unique values, min,
#   max, mean, and standard deviation.
#
# WHY THIS MATTERS:
#   This helps you distinguish:
#     - CONTINUOUS: many unique values, wide range (e.g., LotArea)
#     - DISCRETE: few unique values, integer counts (e.g., FullBath)
#     - ORDINAL: small fixed scale with meaningful order (e.g., OverallQual 1-10)
#     - CATEGORICAL CODE: numeric codes that are labels, not quantities (e.g., MSSubClass)
#
# WHAT TO LOOK FOR:
#   - Columns with <= 25 unique values: probably discrete or categorical
#   - Columns with exactly 2 unique values: binary
#   - Columns with values like 1, 2, 3, 4, 5: possible ordinal ratings
#   - Year columns: special — integers representing time, not counts

numeric_summary = []

for col in numeric_cols:
    n_unique = train_df[col].nunique()
    min_val = train_df[col].min()
    max_val = train_df[col].max()
    mean_val = train_df[col].mean()
    std_val = train_df[col].std()

    numeric_summary.append({
        'column': col,
        'dtype': train_df[col].dtype,
        'n_unique': n_unique,
        'min': min_val,
        'max': max_val,
        'mean': round(mean_val, 2),
        'std': round(std_val, 2)
    })

numeric_summary_df = pd.DataFrame(numeric_summary)
numeric_summary_df = numeric_summary_df.sort_values('n_unique')

print("=== NUMERIC COLUMNS SORTED BY NUMBER OF UNIQUE VALUES ===")
print("(Low unique values often mean discrete/categorical/ordinal)")
display(numeric_summary_df)


=== NUMERIC COLUMNS SORTED BY NUMBER OF UNIQUE VALUES ===
(Low unique values often mean discrete/categorical/ordinal)


,column,dtype,n_unique,min,max,mean,std
19,HalfBath,int64,3,0.0,2.0,0.38,0.50
17,BsmtHalfBath,int64,3,0.0,2.0,0.06,0.24
18,FullBath,int64,4,0.0,3.0,1.57,0.55
16,BsmtFullBath,int64,4,0.0,3.0,0.43,0.52
23,Fireplaces,int64,4,0.0,3.0,0.61,0.64
21,KitchenAbvGr,int64,4,0.0,3.0,1.05,0.22
25,GarageCars,int64,5,0.0,4.0,1.77,0.75
35,YrSold,int64,5,2006.0,2010.0,2007.82,1.33
32,PoolArea,int64,8,0.0,738.0,2.76,40.18
20,BedroomAbvGr,int64,8,0.0,8.0,2.87,0.82


In [ ]:
#   CONTINUOUS: Numeric measurements with many possible values.
#               Example: LotArea, GrLivArea, SalePrice.



#   DISCRETE:   Numeric counts or integers with a limited set of values.
#               Example: FullBath, BedroomAbvGr, GarageCars.
HalfBath
BsmtHalfBath



#   ORDINAL:    Categories with a meaningful order.
#               Example: OverallQual (1-10), KitchenQual (Ex>Gd>TA>Fa>Po).




#   NOMINAL:    Categories with no meaningful order.
#               Example: Neighborhood, MSZoning, MSSubClass.





